# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
email = "pelayosierra@protonmail.com"
EXPERIMENT_NAME = f"/Users/{email}/2-entrena-tu-primer-modelo"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
2026/10/02 07:31:05 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.


In [0]:
@mlflow.trace
def route_question(question: str) -> str:
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/10/02 07:33:37 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-56aacb5a7f6c720aca303b75e2febae8), Trace(trace_id=tr-282a7244bc453bfb05ba7081314417b8), Trace(trace_id=tr-b8d7007fdaa986ff875f53333598b3f2)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.

---

Selecciono la primera traza, {"question": "¿Qué objeto agrupa varios runs?"}.
La ruta seguida por la pregunta, a través de route_question, es mlflow.
El contexto recuperado, es decir, el output a través de retrieve_course_context, fue en este caso "Un experimento agrupa runs con parámetros, métricas y artefactos.".

En caso de que una ruta fuera incorrecta, la solución sería añadir o ajustar las palabras clave en la condición correspondiente de route_question.



## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    {"question": "¿Es seguro poner mi token en un parámetro de MLflow?", "must_include": "secretos"},
    {"question": "¿Qué necesito para monitorizar un servicio en producción?", "must_include": "producción"},
]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Es seguro poner mi token en un parámetro de MLflow?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué necesito para monitorizar un servicio en producción?,producción,"Ruta: production. En producción importan contratos, versionado y monitorización.",true


[Trace(trace_id=tr-e95a8904f83271ffe5d0dabc49ef3a48), Trace(trace_id=tr-7b199cdc86b1e8395b1e333c12949e6a), Trace(trace_id=tr-5615a065256b9a1792b62bc33212baed), Trace(trace_id=tr-48d2d91b1308f62036f584fb39aed556), Trace(trace_id=tr-e91d0ff39987a39a182798c78d844ac2)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.

---

Tras verificarlo:
- pass_rate = 1
- case_count = 5
- results.json:
```json
[
  {
    "question": "\u00bfQu\u00e9 objeto agrupa varios runs?",
    "must_include": "experimento",
    "answer": "Ruta: mlflow. Un experimento agrupa runs con par\u00e1metros, m\u00e9tricas y artefactos.",
    "passed": true
  },
  {
    "question": "\u00bfPuedo guardar una contrase\u00f1a como tag?",
    "must_include": "secretos",
    "answer": "Ruta: security. No registres secretos en par\u00e1metros, tags ni artefactos.",
    "passed": true
  },
  {
    "question": "\u00bfQu\u00e9 debo cuidar en producci\u00f3n?",
    "must_include": "contratos",
    "answer": "Ruta: production. En producci\u00f3n importan contratos, versionado y monitorizaci\u00f3n.",
    "passed": true
  },
  {
    "question": "\u00bfEs seguro poner mi token en un par\u00e1metro de MLflow?",
    "must_include": "secretos",
    "answer": "Ruta: security. No registres secretos en par\u00e1metros, tags ni artefactos.",
    "passed": true
  },
  {
    "question": "\u00bfQu\u00e9 necesito para monitorizar un servicio en producci\u00f3n?",
    "must_include": "producci\u00f3n",
    "answer": "Ruta: production. En producci\u00f3n importan contratos, versionado y monitorizaci\u00f3n.",
    "passed": true
  }
]
```


In [0]:
mlflow.end_run()
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/10/02 07:53:29 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


{'evaluation_run_id': '31240b60ee97403a82b13546a7446e9f', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.

---

Los resultados de la ejecución son los esperados

In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
31240b60ee97403a82b13546a7446e9f,v1-deterministic,1.0
b5d2a8afd2044d608b2769bf596cb3da,v1-deterministic,null


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente

---
Captura:
![image_1790928768939.png](./image_1790928768939.png "image_1790928768939.png")

El evaluation ID, visible en la celda anterior, para sería `31240b60ee97403a82b13546a7446e9f`

`pass_rate` = 1.0, los 5 casos pasan.

En la siguiente versión cubriría el riesgo de enrutado ambiguo. Por ejemplo, una pregunta que mencione tanto 'token' como 'producción' podría ir a security o production según el orden de las condiciones, así que añadiría priorización o puntuación de rutas.


